In [1]:
# ============================================================
# Churn Scoring Pipeline
# AI for Fintech | [RISK-001]
# ============================================================

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))
print(f'Path: {os.path.abspath("..")}')
print('✅ Setup complete')

Path: c:\Users\datav\Desktop\AI_Fintech\Churn
✅ Setup complete


In [3]:
# EDA
# Reads the raw CSV, generates summary report, saves clean parquet

from src.eda import EDA

eda = EDA(
    file_path='../data/bank_customer_churn.csv',
    target_column='churn',
    columns_to_exclude=['customer_id']
)
eda.run()


EDA REPORT
Source: ../data/bank_customer_churn.csv

DATASET SHAPE
Rows:    10000
Columns: 11

COLUMN TYPES
int64      7 columns
str        2 columns
float64    2 columns

NULL VALUES
No null values found.

DUPLICATES
0 duplicated rows found.

CATEGORICAL DISTRIBUTION

country:
  France          5014  (50.1%)
  Germany         2509  (25.1%)
  Spain           2477  (24.8%)

gender:
  Male            5457  (54.6%)
  Female          4543  (45.4%)

NUMERICAL SUMMARY
                    count           mean       25%         50%          75%        max
credit_score      10000.0     650.528800    584.00     652.000     718.0000     850.00
age               10000.0      38.921800     32.00      37.000      44.0000      92.00
tenure            10000.0       5.012800      3.00       5.000       7.0000      10.00
balance           10000.0   76485.889288      0.00   97198.540  127644.2400  250898.09
products_number   10000.0       1.530200      1.00       1.000       2.0000       4.00
credit_card

In [4]:
# Target Analysis
# Validates target distribution and balance

from src.target_analysis import TargetAnalysis

target_analysis = TargetAnalysis(
    file_path='../data/churn_clean.parquet'
)
target_analysis.run()


TARGET ANALYSIS REPORT
Source: ../data/churn_clean.parquet

TARGET — y
Type:          int64
Unique values: [np.int64(0), np.int64(1)]

Null values:   0

Distribution:
  Class 0:  7963  (79.6%)
  Class 1:  2037  (20.4%)

Balance check:
  ⚠️  Imbalanced — minority class at 20.4%
  Recommendation: use class_weight or resampling strategy

Target analysis complete.


In [5]:
# Preprocessing
# Label encoding for categoricals, standard scaling for numericals

from src.preprocessing import Preprocessing

preprocessing = Preprocessing(
    file_path='../data/churn_clean.parquet'
)
preprocessing.run()


PREPROCESSING
Source: ../data/churn_clean.parquet

Transformations:
  ✅ target separated
  ✅ encoded: country
  ✅ encoded: gender
  ✅ scaled: 8 numerical columns
  ✅ target reattached

✅ Preprocessed dataset saved to ../data/churn_preprocessed.parquet
   Total columns: 11
  ✅ encoders saved to ../models/label_encoders.pkl
  ✅ scaler saved to ../models/scaler.pkl

Preprocessing complete.


In [6]:
# Célula 6 — Feature Engineering
# Generates candidate features automatically via PolynomialFeatures

from src.feature_engineering import FeatureEngineering

feature_engineering = FeatureEngineering(
    file_path='../data/churn_preprocessed.parquet',
    degree=2
)
feature_engineering.run()


FEATURE ENGINEERING
Source: ../data/churn_preprocessed.parquet
Degree: 2

Generating features:
  ✅ target separated
  ✅ 55 new features generated
  ✅ target reattached

✅ Feature matrix saved to ../data/churn_features.parquet
   Total features: 66
  ✅ polynomial transformer saved to ../models/polynomial.pkl

Feature engineering complete.


In [7]:
# Feature Selection
# Ranks features by RandomForest importance, selects top N

from src.feature_selection import FeatureSelection

feature_selection = FeatureSelection(
    file_path='../data/churn_features.parquet',
    max_features=20
)
feature_selection.run()


FEATURE SELECTION
Source:       ../data/churn_features.parquet
Max features: 20

Selecting features:
  ✅ target separated
  ✅ features ranked by importance
  ✅ top 20 features selected

Selected features:
  01. age                                      0.0747
  02. products_number^2                        0.0423
  03. products_number active_member            0.0394
  04. balance products_number                  0.0380
  05. age active_member                        0.0320
  06. age products_number                      0.0319
  07. country age                              0.0287
  08. gender age                               0.0258
  09. age balance                              0.0247
  10. products_number                          0.0238
  11. balance^2                                0.0191
  12. balance                                  0.0190
  13. estimated_salary^2                       0.0185
  14. balance active_member                    0.0182
  15. tenure estimated_salary         

In [8]:
# Training
# XGBoost with Bayesian hyperparameter optimization via Optuna

from src.training import Training

training = Training(
    file_path='../data/churn_selected.parquet'
)
training.run()

c:\Users\datav\Desktop\AI_Fintech\Churn\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



TRAINING
Source:  ../data/churn_selected.parquet
Trials:  50
Folds:   5

Optimizing:
  ✅ target separated
  Running 50 trials...
  ✅ best AUC-ROC: 0.8651

  Best parameters:
    max_depth                 4
    learning_rate             0.013463776069110239
    n_estimators              686
    subsample                 0.9554118995102627
    colsample_bytree          0.6722609900416876
    min_child_weight          7
    gamma                     2.9388426553016163
    scale_pos_weight          3.9091801669121256
    eval_metric               auc
    random_state              42

  ✅ model trained on full dataset

✅ Model saved to ../models/xgb_churn.pkl

Training complete.


In [9]:
# Evaluation
# AUC-ROC, Gini, KS statistic and LIFT curve by decile

from src.evaluation import Evaluation

evaluation = Evaluation(
    file_path='../data/churn_selected.parquet',
    model_path='../models/xgb_churn.pkl'
)
evaluation.run()


EVALUATION
Source: ../data/churn_selected.parquet
Model:  ../models/xgb_churn.pkl

Preparing:
  ✅ data prepared

METRICS
  AUC-ROC: 0.9053
  Gini:    0.8106
  KS:      0.6451

LIFT BY DECILE
  Decile     Churners     Churn Rate     Lift
  10        178         89.00%          4.37x
  9         96          48.00%          2.36x
  8         54          27.00%          1.33x
  7         28          14.00%          0.69x
  6         26          13.00%          0.64x
  5         12          6.00%          0.29x
  4         7           3.50%          0.17x
  3         4           2.00%          0.10x
  2         2           1.00%          0.05x
  1         0           0.00%          0.00x

  ✅ KS curve saved to ../outputs/ks_curve.png
  ✅ LIFT curve saved to ../outputs/lift_curve.png

Evaluation complete.


In [10]:
# Explainability
# SHAP values — global feature importance and single prediction drivers

from src.explainability import Explainability

explainability = Explainability(
    file_path='../data/churn_selected.parquet',
    model_path='../models/xgb_churn.pkl'
)
explainability.run()


EXPLAINABILITY
Source: ../data/churn_selected.parquet
Model:  ../models/xgb_churn.pkl

Computing SHAP values:
  ✅ data prepared
  ✅ SHAP values computed for 2000 samples

TOP 10 FEATURES BY SHAP VALUE
  01. age                                      0.5964
  02. products_number^2                        0.4340
  03. products_number active_member            0.2621
  04. country balance                          0.2544
  05. balance products_number                  0.2338
  06. gender age                               0.2332
  07. age active_member                        0.1282
  08. age products_number                      0.0688
  09. products_number                          0.0626
  10. country age                              0.0610

SINGLE PREDICTION EXPLANATION — Sample 0
  Churn probability: 7.59%

  Top drivers:
  products_number^2                        decreases churn risk  (-0.6524)
  age                                      decreases churn risk  (-0.4046)
  balance products_numb